# Explainable Machine Learning, Hands-On 🔍

## Local & Global Explanations of an Image Classifier

> *“Correct answers ≠ Intelligent.”* — a model can be right without us understanding **why**.

Based on Hung-yi Lee's **Explainable ML** lecture.

### Why explainability?

A deep network can hit high accuracy yet stay a **black box**. But many domains — **loan approval, medical diagnosis, court decisions, self-driving cars** — legally or ethically *require* a reason for every decision. We cannot just deploy the most accurate model and walk away.

This is the **interpretable-vs-powerful trade-off**: linear and tree models are easy to read but limited; deep nets are powerful but opaque. Explainable ML aims for the best of both — keep the powerful model, but extract human-readable explanations from it.

### The organizing taxonomy

Every method below answers one of two questions:

- **LOCAL explanation** — *“Why is THIS image a 7?”* (which components of one input drove the prediction)
- **GLOBAL explanation** — *“What does a 7 look like to the model?”* (what input most excites a class, independent of any single image)

### Roadmap

We train **one small CNN on MNIST** in-notebook and run every method on it, end-to-end, on a free Colab CPU:

| Section | Method | Type | Objective |
|---|---|---|---|
| 1 | **Occlusion sensitivity** | local | LO2 |
| 2 | **Saliency map** (gradients) | local | LO3 |
| 3 | **SmoothGrad** (noisy-gradient fix) | local | LO4 |
| 4 | **Integrated Gradients** (saturation fix) | local | LO4 |
| 5 | **Clever-Hans** case study | diagnosis | LO5 |
| 6 | **Activation maximization** | global | LO6 |
| 7 | **LIME** (model-agnostic) | local | LO7 |

Each section has a short explanation, a from-scratch implementation, and an interactive widget. Run the cells top to bottom.

## Part 0 — Setup

This demo uses **PyTorch + torchvision** with **MNIST**: a small, built-in dataset whose 28×28 digit images train fast and make the activation-maximization “class visualizations” directly comparable to the lecture's digit examples.

The “black box” we explain is a **small CNN trained in-notebook**. Everything runs on a **free Colab CPU** in seconds-to-minutes — no GPU required.

- The only install is **`ipywidgets`** (we implement LIME from scratch, so `scikit-image` is optional and unused).
- Random seeds are **fixed** for reproducibility.
- Every interactive cell also renders a **sensible static default**, so the notebook executes end-to-end even if widgets fail to initialize.

In [ ]:
# Part 0 — Environment, imports, reproducibility
# torch, torchvision, numpy, matplotlib are preinstalled on Colab; we only need ipywidgets.
try:
    get_ipython().run_line_magic('pip', 'install -q ipywidgets')
except Exception as e:
    print(f'[setup] ipywidgets install skipped/failed ({e}); interactive cells fall back to static defaults.')

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, TensorDataset
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox

# Inline plotting (guarded so the cell also runs under plain python).
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

SEED = 42

def set_seed(seed: int = 42) -> None:
    '''Seed python / numpy / torch (and CUDA if present) for reproducible runs.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Matplotlib defaults tuned for small grayscale digit images.
plt.rcParams['figure.figsize'] = (8, 3)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 110

print(f'torch {torch.__version__} | device: {device} | seed: {SEED}')

## The shared substrate

Before any method, fix the setup the whole notebook reuses:

- **Task:** classify 28×28 MNIST digits into **10 classes** with a small CNN.
- **Components:** treat any input as components $x_1, \dots, x_N$ — here, **pixels** (for gradient methods) or **patches / superpixels** (for occlusion and LIME). A *local* explanation scores each component's **importance**.

The rest of the notebook answers two questions about **one trained model**:

- **LOCAL** — *“Which components of THIS image drove the prediction?”* → occlusion, saliency, SmoothGrad, integrated gradients, LIME.
- **GLOBAL** — *“What input most activates class $c$, independent of any single image?”* → activation maximization.

The **same `clean_model`** is reused everywhere, so explanations are directly comparable across methods.

In [ ]:
# Load MNIST once and define the sampling helper.
transform = transforms.ToTensor()  # -> float tensors in [0,1], shape (1,28,28)

def _load_mnist():
    train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    return train, test

try:
    full_train, test_dataset = _load_mnist()
except Exception as e:
    print(f'[data] MNIST download failed once ({e}); retrying...')
    full_train, test_dataset = _load_mnist()

# Subsample the training set so in-notebook training stays fast on CPU.
SUBSET_SIZE = 10000
BATCH_SIZE = 128
_gen = torch.Generator().manual_seed(SEED)
_perm = torch.randperm(len(full_train), generator=_gen)[:SUBSET_SIZE]
train_subset = Subset(full_train, _perm.tolist())

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

class_names = [str(i) for i in range(10)]

def get_sample(index: int, dataset: str = 'test'):
    '''Return a single (image_tensor (1,28,28), int label) pair, clamping index into range.'''
    if dataset not in ('test', 'train'):
        print('[get_sample] unknown dataset, defaulting to test.')
        dataset = 'test'
    ds = test_dataset if dataset == 'test' else train_subset
    index = max(0, min(int(index), len(ds) - 1))
    img, label = ds[index]
    return img, int(label)

# Show a few example digits so students see the data.
fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for i, ax in enumerate(axes):
    img, lab = get_sample(i, 'test')
    ax.imshow(img.squeeze(0).numpy(), cmap='gray')
    ax.set_title(class_names[lab])
    ax.axis('off')
plt.suptitle('MNIST test samples')
plt.tight_layout()
plt.show()

print(f'train subset: {len(train_subset)} | test: {len(test_dataset)}')

In [ ]:
# Define and train the shared 'black box' model.
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # -> (N,16,14,14)
        x = self.pool(F.relu(self.conv2(x)))   # -> (N,32,7,7)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)                      # raw logits (N,10)

EPOCHS = 2

def train_model(model, loader, epochs: int = 2, lr: float = 1e-3):
    set_seed(SEED)  # reproducible init / data order
    model.to(device).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for ep in range(epochs):
        running = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * x.size(0)
        print(f'  epoch {ep + 1}/{epochs} | loss {running / len(loader.dataset):.4f}')
    return model

def evaluate_accuracy(model, loader) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total

def _as_batch(image) -> torch.Tensor:
    '''Normalize (28,28)/(1,28,28)/(1,1,28,28) -> (1,1,28,28) float tensor on device.'''
    if not torch.is_tensor(image):
        image = torch.as_tensor(image)
    image = image.float()
    if image.dim() == 2:
        image = image.unsqueeze(0).unsqueeze(0)
    elif image.dim() == 3:
        image = image.unsqueeze(0)
    elif image.dim() != 4:
        raise ValueError(f'_as_batch: unexpected rank {image.dim()} for shape {tuple(image.shape)}')
    if tuple(image.shape[-2:]) != (28, 28):
        raise ValueError(f'_as_batch: expected trailing (28,28), got shape {tuple(image.shape)}')
    return image.to(device)

def predict_proba(model, image) -> np.ndarray:
    '''Softmax class-probability vector (10,) as numpy; the model-query primitive for occlusion and LIME.'''
    x = _as_batch(image)
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)[0]
    return probs.detach().cpu().numpy()

set_seed(SEED)
clean_model = SmallCNN()
print('Training clean_model on MNIST...')
train_model(clean_model, train_loader, epochs=EPOCHS)
clean_model.eval()
train_acc = evaluate_accuracy(clean_model, train_loader)
test_acc = evaluate_accuracy(clean_model, test_loader)
print(f'clean_model | train acc {train_acc:.3f} | test acc {test_acc:.3f}')
if test_acc < 0.90:
    print('[warn] test accuracy < 0.90 — consider one more epoch (raise EPOCHS).')

## 1 — Occlusion sensitivity (gradient-free)

The most intuitive local explanation, and a perfect warm-up because it uses **no gradient math at all**.

**The rule (from the slides):** take the components $x_1, \dots, x_N$, **remove or modify** one of them, and watch the model's score for the predicted class.

- Here we slide a small **gray / zero patch** over each region of the image.
- If covering a region makes the predicted-class probability **drop a lot**, that region was **important**.
- Collect the drops into a 2D **importance heatmap**.

This needs only **forward passes** (no autograd) and follows the gray-patch occlusion experiments of **Zeiler & Fergus (2014)**.

In [ ]:
# Gradient-free occlusion sensitivity.
def occlusion_heatmap(model, image, patch_size: int = 7, stride: int = 2, occlusion_value: float = 0.0) -> np.ndarray:
    '''Slide an occluding patch; record the predicted-class probability drop -> (28,28) heatmap.'''
    patch_size = int(max(1, min(patch_size, 28)))
    stride = int(max(1, min(stride, 28)))
    base_probs = predict_proba(model, image)
    target = int(base_probs.argmax())
    base_score = float(base_probs[target])

    img2d = _as_batch(image)[0, 0].detach().cpu().numpy()  # (28,28)
    tops = list(range(0, 28, stride))
    lefts = list(range(0, 28, stride))
    coarse = np.zeros((len(tops), len(lefts)), dtype=np.float32)
    for ti, top in enumerate(tops):
        for li, left in enumerate(lefts):
            occ = img2d.copy()
            occ[top:min(top + patch_size, 28), left:min(left + patch_size, 28)] = occlusion_value
            occ_probs = predict_proba(model, torch.from_numpy(occ))
            coarse[ti, li] = base_score - float(occ_probs[target])

    # Upsample the coarse grid to 28x28 (nearest-neighbor) then crop.
    rep_r = int(np.ceil(28 / coarse.shape[0]))
    rep_c = int(np.ceil(28 / coarse.shape[1]))
    heat = np.kron(coarse, np.ones((rep_r, rep_c), dtype=np.float32))[:28, :28]
    return np.clip(heat, 0.0, None)

# Static demo on a fixed sample.
_img, _lab = get_sample(7, 'test')
_hm = occlusion_heatmap(clean_model, _img)
_p = predict_proba(clean_model, _img)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(_img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {_lab})')
axes[1].imshow(_hm, cmap='jet'); axes[1].set_title('occlusion heatmap')
axes[2].imshow(_img.squeeze(0).numpy(), cmap='gray')
axes[2].imshow(_hm, cmap='jet', alpha=0.5); axes[2].set_title('overlay')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'Occlusion — predicted {int(_p.argmax())} (p={_p.max():.2f})')
plt.tight_layout(); plt.show()

In [ ]:
# Interactive occlusion explorer.
def render_occlusion(image_index: int = 7, patch_size: int = 7, stride: int = 2) -> None:
    img, lab = get_sample(image_index, 'test')
    hm = occlusion_heatmap(clean_model, img, patch_size=patch_size, stride=stride)
    p = predict_proba(clean_model, img)
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {lab})')
    axes[1].imshow(hm, cmap='jet'); axes[1].set_title('occlusion heatmap')
    axes[2].imshow(img.squeeze(0).numpy(), cmap='gray')
    axes[2].imshow(hm, cmap='jet', alpha=0.5); axes[2].set_title('overlay')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'patch={patch_size}, stride={stride} | predicted {int(p.argmax())} (p={p.max():.2f})')
    plt.tight_layout(); plt.show()

# Static default FIRST so a figure always appears, even without widgets.
render_occlusion()

try:
    interact(render_occlusion,
             image_index=IntSlider(7, 0, 200, 1),
             patch_size=IntSlider(7, 3, 14, 1),
             stride=IntSlider(2, 1, 7, 1))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 2 — Saliency map (gradient-based)

The **centerpiece** local-explanation method. Where occlusion *perturbs and re-runs*, saliency reads importance **directly from the gradient**.

For class score $e$ and pixel $x_n$:

$$\text{importance}(x_n) = \left| \frac{\partial e}{\partial x_n} \right|$$

obtained from a **single backward pass**.

**vs occlusion:**

- **Cheaper** — one backward pass instead of hundreds of forward passes.
- **Higher resolution** — per-pixel instead of per-patch.

Reference: **Simonyan, Vedaldi & Zisserman (ICLR 2014)**, *“Deep Inside Convolutional Networks.”*

In [ ]:
# Gradient-based saliency.
def saliency_map(model, image, target_class=None) -> np.ndarray:
    '''importance = |d(class score)/d(pixel)| from one backward pass -> (28,28) normalized to [0,1].'''
    x = _as_batch(image).clone().detach().requires_grad_(True)
    model.eval()
    logits = model(x)
    if target_class is None:
        target_class = int(logits.argmax(dim=1).item())
    target_class = int(max(0, min(target_class, 9)))
    model.zero_grad(set_to_none=True)
    logits[0, target_class].backward()
    grad = x.grad.detach().abs()[0]            # (1,28,28)
    sal = grad.max(dim=0)[0].cpu().numpy()     # reduce over channels -> (28,28)
    m = sal.max()
    return sal / m if m > 0 else sal

# Static demo: occlusion (gradient-free) vs saliency (gradient) on the same sample.
_img, _lab = get_sample(7, 'test')
_occ = occlusion_heatmap(clean_model, _img)
_sal = saliency_map(clean_model, _img)
_p = predict_proba(clean_model, _img)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(_img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {_lab})')
axes[1].imshow(_occ, cmap='jet'); axes[1].set_title('occlusion (gradient-free)')
axes[2].imshow(_sal, cmap='hot'); axes[2].set_title('saliency (gradient)')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'predicted {int(_p.argmax())} (p={_p.max():.2f})')
plt.tight_layout(); plt.show()

In [ ]:
# Interactive saliency explorer (supports the instructive 'why NOT this class?' case).
_CMAPS = ['hot', 'viridis', 'gray', 'jet']

def render_saliency(image_index: int = 7, target_class: str = 'predicted', colormap: str = 'hot') -> None:
    img, lab = get_sample(image_index, 'test')
    tc = None if str(target_class) == 'predicted' else int(target_class)
    if colormap not in _CMAPS:
        colormap = 'hot'
    sal = saliency_map(clean_model, img, tc)
    p = predict_proba(clean_model, img)
    shown = 'predicted' if tc is None else str(tc)
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {lab})')
    axes[1].imshow(sal, cmap=colormap); axes[1].set_title(f'saliency for class {shown}')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'predicted {int(p.argmax())} (p={p.max():.2f}) | explaining class {shown}')
    plt.tight_layout(); plt.show()

render_saliency()

try:
    interact(render_saliency,
             image_index=IntSlider(7, 0, 200, 1),
             target_class=Dropdown(options=['predicted'] + [str(i) for i in range(10)], value='predicted'),
             colormap=Dropdown(options=_CMAPS, value='hot'))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 3 — SmoothGrad (fixing noisy gradients)

Raw saliency maps look **speckled** — saliency's first stated limitation, **“Noisy Gradient.”** The gradient fluctuates rapidly from pixel to pixel.

**SmoothGrad** is a tiny refinement layered right on top of the saliency function:

1. Add **Gaussian noise** to the input.
2. Compute a saliency map for that noisy copy.
3. **Repeat n times and average.**

Averaging cancels the noise and leaves a far **cleaner** attribution. Two knobs: the **noise level σ** and the **sample count n**.

Reference: **Smilkov et al. (2017)**, arXiv:1706.03825.

In [ ]:
# SmoothGrad: average saliency over noise-perturbed copies.
def smoothgrad(model, image, target_class=None, noise_level: float = 0.15, n_samples: int = 25) -> np.ndarray:
    n_samples = max(1, int(n_samples))
    base = _as_batch(image)
    # Resolve the target class ONCE so every noisy copy is attributed to the SAME class.
    if target_class is None:
        model.eval()
        with torch.no_grad():
            target_class = int(model(base).argmax(dim=1).item())
    target_class = int(max(0, min(target_class, 9)))
    rng = float(base.max() - base.min())
    sigma = noise_level * (rng if rng > 0 else 1.0)
    accum = np.zeros((28, 28), dtype=np.float32)
    for _ in range(n_samples):
        noisy = (base + torch.randn_like(base) * sigma).clamp(0, 1)
        accum += saliency_map(model, noisy, target_class)
    return accum / n_samples

_sg = smoothgrad(clean_model, get_sample(7, 'test')[0], n_samples=10)
print(f'smoothgrad ok | shape {_sg.shape} | target fixed to predicted class | finite={bool(np.isfinite(_sg).all())}')

In [ ]:
# Interactive raw-vs-SmoothGrad comparison.
def render_smoothgrad(image_index: int = 7, noise_level: float = 0.15, n_samples: int = 25) -> None:
    img, lab = get_sample(image_index, 'test')
    raw = saliency_map(clean_model, img)
    sg = smoothgrad(clean_model, img, noise_level=noise_level, n_samples=n_samples)
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {lab})')
    axes[1].imshow(raw, cmap='hot'); axes[1].set_title('raw saliency')
    axes[2].imshow(sg, cmap='hot'); axes[2].set_title('SmoothGrad')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'sigma={noise_level:.2f}, n={n_samples}  (very high noise washes the map out)')
    plt.tight_layout(); plt.show()

render_smoothgrad()

# n_samples capped at 100 so CPU recompute stays interactive (latency vs quality trade-off).
try:
    interact(render_smoothgrad,
             image_index=IntSlider(7, 0, 200, 1),
             noise_level=FloatSlider(0.15, 0.0, 0.5, 0.01),
             n_samples=IntSlider(25, 1, 100, 1))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 4 — Integrated Gradients (fixing gradient saturation)

Saliency's **second** stated limitation: **“Gradient Saturation.”** Once a feature is “large enough,” its gradient can **vanish** even though the feature is decisive.

> *The slides' example:* the gradient ∂(elephant score)/∂(trunk length) ≈ 0 for an image that is *clearly* an elephant — the trunk already saturated the “elephant” score.

**Integrated Gradients** fixes this by accumulating gradients along a straight path from a **baseline** $b$ (e.g. a black image) to the actual input $x$, approximated as a Riemann sum over `steps` points:

$$\text{IG}_n = (x_n - b_n)\times\frac{1}{m}\sum_{k=1}^{m}\frac{\partial e}{\partial x_n}\left(b + \frac{k}{m}(x-b)\right)$$

It also satisfies **completeness**: the attributions sum to the score difference $e(x) - e(b)$.

Reference: arXiv:1611.02639.

In [ ]:
# Integrated Gradients with a completeness check.
def integrated_gradients(model, image, target_class=None, baseline=None, steps: int = 50) -> np.ndarray:
    steps = max(1, int(steps))
    x = _as_batch(image)
    if baseline is None:
        baseline = torch.zeros_like(x)
    else:
        baseline = _as_batch(baseline)
        if tuple(baseline.shape) != tuple(x.shape):
            raise ValueError(f'baseline shape {tuple(baseline.shape)} != input shape {tuple(x.shape)}')
    model.eval()
    if target_class is None:
        with torch.no_grad():
            target_class = int(model(x).argmax(dim=1).item())
    target_class = int(max(0, min(target_class, 9)))

    total_grad = torch.zeros_like(x)
    for k in range(1, steps + 1):
        alpha = k / steps
        interp = (baseline + alpha * (x - baseline)).clone().detach().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        model(interp)[0, target_class].backward()
        total_grad += interp.grad.detach()
    avg_grad = total_grad / steps
    attributions = ((x - baseline) * avg_grad)[0].sum(dim=0).cpu().numpy()  # (28,28) signed

    # Completeness check on the target LOGIT (not softmax prob).
    with torch.no_grad():
        exact = float(model(x)[0, target_class] - model(baseline)[0, target_class])
    approx = float(attributions.sum())
    print(f'IG completeness | sum(attr)={approx:.3f} vs logit(x)-logit(base)={exact:.3f} '
          f'| abs error={abs(approx - exact):.3f} (tightens as steps grows)')
    return attributions

_ig = integrated_gradients(clean_model, get_sample(7, 'test')[0], steps=50)

In [ ]:
# Interactive Integrated Gradients explorer.
def make_baseline(image, baseline_type: str = 'black') -> torch.Tensor:
    x = _as_batch(image)
    if baseline_type not in ('black', 'random', 'blur'):
        print('[make_baseline] unknown type, defaulting to black.')
        baseline_type = 'black'
    if baseline_type == 'black':
        return torch.zeros_like(x)
    if baseline_type == 'random':
        gen = torch.Generator(device='cpu').manual_seed(SEED)
        return torch.rand(x.shape, generator=gen).to(x.device)
    return F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)  # simple box blur

def render_ig(image_index: int = 7, steps: int = 50, baseline_type: str = 'black') -> None:
    img, lab = get_sample(image_index, 'test')
    base = make_baseline(img, baseline_type)
    ig = integrated_gradients(clean_model, img, baseline=base, steps=steps)
    with torch.no_grad():
        xb = _as_batch(img)
        tc = int(clean_model(xb).argmax(dim=1).item())
        exact = float(clean_model(xb)[0, tc] - clean_model(base)[0, tc])
    err = abs(float(ig.sum()) - exact)
    lim = float(np.abs(ig).max()) or 1.0
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {lab})')
    axes[1].imshow(base[0, 0].detach().cpu().numpy(), cmap='gray'); axes[1].set_title(f'baseline: {baseline_type}')
    axes[2].imshow(ig, cmap='bwr', vmin=-lim, vmax=lim); axes[2].set_title('IG attribution')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'steps={steps}, baseline={baseline_type} | completeness err={err:.3f}')
    plt.tight_layout(); plt.show()

render_ig()

try:
    interact(render_ig,
             image_index=IntSlider(7, 0, 200, 1),
             steps=IntSlider(50, 5, 200, 5),
             baseline_type=Dropdown(options=['black', 'random', 'blur'], value='black'))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 5 — Clever-Hans: explanation exposes a shortcut learner

The lecture's pedagogical climax. **Clever Hans** was a horse that appeared to do arithmetic but was really reading its trainer's body language — *right answer, wrong reason*.

**The Pokémon-vs-Digimon case study:** a classifier reached **98.9% train / 98.4% test** accuracy... yet it was secretly discriminating by **background color** — the Pokémon PNGs had transparent→black backgrounds while the Digimon JPEGs did not. Saliency maps gave it away: they lit up the **background**, not the creatures. (The PASCAL VOC “horse” detector keying on a **source watermark** is the same story.)

Those datasets are not redistributable, so we **reproduce the lesson synthetically**: stamp a small bright **corner watermark** whose presence is correlated with the digit's **parity** (even vs odd). A model can then “cheat” by reading the watermark instead of the digit.

**The plan:** train a shortcut model on watermarked data → confirm it looks great on watermarked test data → use **saliency** to reveal it ignores the digit, and watch its accuracy **drop** when the watermark is removed.

In [ ]:
# Build a Clever-Hans (spurious-correlation) dataset and train a shortcut model.
def make_watermarked(images, labels, correlation: float = 0.95, patch_size: int = 4, value: float = 1.0) -> torch.Tensor:
    '''Stamp a bright top-left patch whose presence is ~correlation-correlated with digit parity (even vs odd).'''
    correlation = float(max(0.0, min(correlation, 1.0)))
    patch_size = int(max(1, min(patch_size, 28)))
    if images.dim() == 3:
        images = images.unsqueeze(0)
    out = images.clone()  # never mutate the input dataset tensors
    labels = torch.as_tensor(labels).view(-1)
    gen = torch.Generator().manual_seed(SEED)
    cond = (labels % 2 == 0)                                       # True for even digits
    flips = torch.rand(len(labels), generator=gen) > correlation   # flip with prob (1 - correlation)
    stamp = cond ^ flips                                           # watermark present iff stamp True
    idx = torch.where(stamp)[0]
    out[idx, :, :patch_size, :patch_size] = value
    return out

def evaluate_model(model, loader) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total

# Pull the training subset into memory and watermark it.
_train_imgs = torch.stack([train_subset[i][0] for i in range(len(train_subset))])
_train_lbls = torch.tensor([train_subset[i][1] for i in range(len(train_subset))])
_wm_train = make_watermarked(_train_imgs, _train_lbls, correlation=0.95)
watermarked_train_loader = DataLoader(TensorDataset(_wm_train, _train_lbls), batch_size=BATCH_SIZE, shuffle=True)

# Two test sets: watermark consistent with parity, and watermark removed.
_test_imgs = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
_test_lbls = torch.tensor([test_dataset[i][1] for i in range(len(test_dataset))])
_wm_test = make_watermarked(_test_imgs, _test_lbls, correlation=1.0)
watermarked_test_loader = DataLoader(TensorDataset(_wm_test, _test_lbls), batch_size=256, shuffle=False)
clean_test_loader = DataLoader(TensorDataset(_test_imgs.clone(), _test_lbls), batch_size=256, shuffle=False)

set_seed(SEED)
shortcut_model = SmallCNN()
print('Training shortcut_model on watermarked data...')
train_model(shortcut_model, watermarked_train_loader, epochs=EPOCHS)
shortcut_model.eval()
acc_wm = evaluate_model(shortcut_model, watermarked_test_loader)
acc_clean = evaluate_model(shortcut_model, clean_test_loader)
print(f'shortcut_model | watermark-consistent test acc {acc_wm:.3f} | watermark-REMOVED test acc {acc_clean:.3f}')
print(f'accuracy drop = {acc_wm - acc_clean:.3f}  (large gap => the model leaned on the shortcut)')
if acc_wm - acc_clean < 0.15:
    print('[warn] small gap — try a higher correlation or larger patch so the collapse is clearly visible.')

In [ ]:
# Interactive Clever-Hans reveal: shortcut-model vs clean-model saliency.
def render_clever_hans(image_index: int = 12, add_watermark: bool = True) -> None:
    img, label = get_sample(image_index, 'test')
    # correlation=1.0 here so the checkbox cleanly adds/removes the SAME deterministic patch.
    x = make_watermarked(img.unsqueeze(0), torch.tensor([label]), correlation=1.0)[0] if add_watermark else img
    pred_short = int(predict_proba(shortcut_model, x).argmax())
    pred_clean = int(predict_proba(clean_model, x).argmax())
    s_short = saliency_map(shortcut_model, x)
    s_clean = saliency_map(clean_model, x)
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(x.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {label}, wm={add_watermark})')
    axes[1].imshow(s_short, cmap='hot'); axes[1].set_title(f'shortcut saliency (pred {pred_short})')
    axes[2].imshow(s_clean, cmap='hot'); axes[2].set_title(f'clean saliency (pred {pred_clean})')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'shortcut_model -> {pred_short} | clean_model -> {pred_clean} | true {label}   '
                 f'(shortcut saliency lights the corner watermark)')
    plt.tight_layout(); plt.show()

render_clever_hans()

try:
    interact(render_clever_hans,
             image_index=IntSlider(12, 0, 200, 1),
             add_watermark=Checkbox(value=True, description='add watermark'))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 6 — Activation maximization (global explanation)

Now the **global** half of the taxonomy. Instead of explaining one input, we **synthesize** the input that most excites a class:

$$X^* = \arg\max_X\ y_c$$

found by **gradient ascent on the image** with the model **frozen**.

**The catch:** done naively, $X^*$ looks like **adversarial noise**, not a digit — *“Consider adversarial attack!”* This directly links explainability to adversarial examples.

Adding a **regularizer** $R(X)$ makes the result human-recognizable:

$$X^* = \arg\max_X\ \big(y_c - \lambda R(X)\big)$$

e.g. an **L1 sparsity** penalty $R(X) = \sum_{ij} |X_{ij}|$ or a **total-variation smoothness** penalty. On MNIST the regularized result becomes **digit-like**.

Reference: **Yosinski et al. (2015)**, arXiv:1506.06579.

In [ ]:
# Activation maximization (global explanation) via gradient ascent, with regularization.
def total_variation(img: torch.Tensor) -> torch.Tensor:
    '''Sum of absolute neighbor differences along H and W (smoothness penalty).'''
    dh = (img[..., 1:, :] - img[..., :-1, :]).abs().sum()
    dw = (img[..., :, 1:] - img[..., :, :-1]).abs().sum()
    return dh + dw

def activation_maximization(model, target_class: int, steps: int = 200, lr: float = 0.1,
                            reg_strength: float = 0.01, reg_type: str = 'l1') -> np.ndarray:
    target_class = int(max(0, min(target_class, 9)))
    steps = max(1, int(steps))
    lr = max(1e-4, float(lr))
    if reg_type not in ('l1', 'tv', 'none'):
        print('[activation_maximization] unknown reg_type, defaulting to l1.')
        reg_type = 'l1'
    # Freeze the model so gradient ascent updates ONLY the synthesized image.
    for p in model.parameters():
        p.requires_grad_(False)
    model.eval()
    set_seed(SEED)
    X = (0.1 * torch.randn(1, 1, 28, 28, device=device)).clamp(0, 1).requires_grad_(True)
    optimizer = torch.optim.Adam([X], lr=lr)
    for _ in range(steps):
        score = model(X)[0, target_class]
        if reg_strength == 0 or reg_type == 'none':
            reg = torch.zeros((), device=device)
        elif reg_type == 'l1':
            reg = X.abs().sum()
        else:
            reg = total_variation(X)
        loss = -score + reg_strength * reg          # maximize score, penalize complexity
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            X.clamp_(0, 1)
    return X.detach()[0, 0].cpu().numpy()

# Demo the lecture's point: same class with no reg (adversarial-looking) vs L1 reg (digit-like).
_cls = 3
_noisy = activation_maximization(clean_model, _cls, steps=200, reg_strength=0.0)
_reg = activation_maximization(clean_model, _cls, steps=200, reg_strength=0.02, reg_type='l1')
_sn = predict_proba(clean_model, torch.from_numpy(_noisy))[_cls]
_sr = predict_proba(clean_model, torch.from_numpy(_reg))[_cls]
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(_noisy, cmap='gray'); axes[0].set_title(f'class {_cls}, no reg (score {_sn:.3f})')
axes[1].imshow(_reg, cmap='gray'); axes[1].set_title(f'class {_cls}, L1 reg (score {_sr:.3f})')
for ax in axes:
    ax.axis('off')
plt.suptitle('Activation maximization: noise -> digit as regularization increases')
plt.tight_layout(); plt.show()

In [ ]:
# Interactive activation-maximization explorer.
def render_activation_max(target_class: int = 3, reg_strength: float = 0.01, steps: int = 200, lr: float = 0.1) -> None:
    tc = int(target_class)
    img = activation_maximization(clean_model, tc, steps=int(steps), lr=float(lr),
                                  reg_strength=float(reg_strength), reg_type='l1')
    score = predict_proba(clean_model, torch.from_numpy(img))[tc]
    plt.figure(figsize=(3.4, 3.6))
    plt.imshow(img, cmap='gray')
    plt.title(f'class {tc} | score {score:.3f} | reg {reg_strength:.3f}')
    plt.axis('off')
    plt.show()

render_activation_max()

# steps capped at 400 so each interactive redraw stays responsive on CPU.
try:
    interact(render_activation_max,
             target_class=IntSlider(3, 0, 9, 1),
             reg_strength=FloatSlider(0.01, 0.0, 0.1, 0.005),
             steps=IntSlider(200, 20, 400, 20),
             lr=FloatSlider(0.1, 0.01, 0.5, 0.01))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## 7 — LIME (model-agnostic local explanation)

The capstone, and it ties right back to the **interpretable-vs-powerful** theme. **LIME** = *Local Interpretable Model-agnostic Explanations*. It explains **any** classifier using only its input/output behavior — **no gradients, no weights**.

**The recipe:**

1. Pick **one** input.
2. Split it into **interpretable components** — here, square **superpixels** (a grid of patches).
3. Generate many **perturbations** that turn components **on / off**.
4. **Query the black box** for each perturbation's predicted-class probability.
5. Fit a simple **weighted linear model** (weighting perturbations by similarity to the original) that mimics the black box **locally**.
6. Read each component's importance off its **linear weight**.

Because it only queries inputs and outputs, LIME works for a neural net, a random forest, or any opaque model. Reference: **Ribeiro et al. (2016)**, *“Why Should I Trust You?”*

In [ ]:
# LIME from scratch — no external lime package. (skimage SLIC is an optional upgrade; we stay dependency-light.)
def grid_segments(image, grid: int = 7) -> np.ndarray:
    '''Partition 28x28 into grid x grid square superpixels -> (28,28) int segment-id map.'''
    grid = int(max(2, min(grid, 14)))
    cell = int(np.ceil(28 / grid))
    n_col_blocks = int(np.ceil(28 / cell))
    rows = (np.arange(28) // cell).reshape(-1, 1)
    cols = (np.arange(28) // cell).reshape(1, -1)
    seg = rows * n_col_blocks + cols
    # Re-index to a dense 0..K-1 range (grid may not divide 28 evenly).
    uniq = {v: i for i, v in enumerate(sorted(set(seg.flatten().tolist())))}
    seg = np.vectorize(uniq.get)(seg).astype(int)
    return seg

def lime_explain(model, image, n_samples: int = 150, grid: int = 7,
                 kernel_width: float = 0.25, baseline_fill: float = 0.0):
    n_samples = max(1, int(n_samples))
    seg = grid_segments(image, grid)
    n_seg = int(seg.max()) + 1
    img2d = _as_batch(image)[0, 0].detach().cpu().numpy()
    predicted = int(predict_proba(model, image).argmax())   # fix the explained class for ALL perturbations

    rng = np.random.RandomState(SEED)
    masks = rng.randint(0, 2, size=(n_samples, n_seg)).astype(np.float32)
    masks[0, :] = 1.0                                        # always include the full-image perturbation
    y = np.zeros(n_samples, dtype=np.float32)
    for i in range(n_samples):
        perturbed = img2d.copy()
        off = np.where(masks[i] == 0)[0]
        if off.size:
            perturbed[np.isin(seg, off)] = baseline_fill     # switch OFF segments to the baseline fill
        y[i] = predict_proba(model, torch.from_numpy(perturbed))[predicted]

    frac_on = masks.mean(axis=1)
    weights = np.exp(-((1.0 - frac_on) ** 2) / (kernel_width ** 2))   # proximity to the original
    W = np.diag(weights)
    lam = 1e-3                                               # ridge term for numerical stability
    A = masks.T @ W @ masks + lam * np.eye(n_seg)
    b = masks.T @ W @ y
    beta = np.linalg.solve(A, b)                             # per-segment importance weights

    order = np.argsort(beta)[::-1][:5]
    print(f'LIME | predicted class {predicted} | top segments {order.tolist()} '
          f'| weights {[round(float(beta[j]), 4) for j in order]}')
    return beta, seg

_w, _seg = lime_explain(clean_model, get_sample(7, 'test')[0], n_samples=100, grid=7)
print(f'segments: {int(_seg.max()) + 1} | weight vector length: {_w.shape[0]}')

In [ ]:
# Interactive LIME explorer.
def render_lime(image_index: int = 7, n_samples: int = 150, grid: int = 7, top_k: int = 5) -> None:
    img, lab = get_sample(image_index, 'test')
    w, seg = lime_explain(clean_model, img, n_samples=int(n_samples), grid=int(grid))
    n_seg = int(seg.max()) + 1
    k = int(max(1, min(top_k, n_seg)))
    top = np.argsort(w)[::-1][:k]
    overlay = np.isin(seg, top).astype(float)
    p = predict_proba(clean_model, img)
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(img.squeeze(0).numpy(), cmap='gray'); axes[0].set_title(f'input (label {lab})')
    axes[1].imshow(img.squeeze(0).numpy(), cmap='gray')
    axes[1].imshow(overlay, cmap='Greens', alpha=0.5); axes[1].set_title(f'top-{k} LIME segments')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'predicted {int(p.argmax())} (p={p.max():.2f}) | grid={grid}, n={n_samples}')
    plt.tight_layout(); plt.show()

render_lime()

# n_samples capped at 400 so interactive LIME stays responsive on CPU.
try:
    interact(render_lime,
             image_index=IntSlider(7, 0, 200, 1),
             n_samples=IntSlider(150, 30, 400, 10),
             grid=IntSlider(7, 3, 12, 1),
             top_k=IntSlider(5, 1, 15, 1))
except Exception as e:
    print(f'[widgets] interact unavailable ({e}); the static figure above is valid.')

## Recap & where to go next

We walked one continuous spine on a **single MNIST CNN** and sorted every method into the taxonomy:

**LOCAL explanation** — *“Why is THIS input class $c$?”*
- **Occlusion** (LO2) — perturb a patch, watch the score drop. Gradient-free, patch-resolution.
- **Saliency** (LO3) — $|\partial e / \partial x|$ in one backward pass. Cheap, pixel-resolution.
- **SmoothGrad** (LO4) — average saliency over noisy copies → fixes **noisy gradients**.
- **Integrated Gradients** (LO4) — accumulate gradients from a baseline → fixes **saturation**, satisfies completeness.
- **LIME** (LO7) — perturb superpixels, fit a local linear surrogate. Model-agnostic.

**GLOBAL explanation** — *“What does class $c$ look like to the model?”*
- **Activation maximization** (LO6) — gradient-ascend the input; regularization turns noise into a digit.

**The key lesson (LO5):** high accuracy can hide a **spurious shortcut**. Our Clever-Hans model aced watermarked test data, then dropped when the watermark vanished — and **saliency exposed it** keying on the corner, not the digit.

### Concepts left out (to keep this fast & self-contained)

- **Attention as explanation** and the *“is attention explanation?”* debate — arXiv:1902.10186, arXiv:1908.04626.
- **Probing classifiers** on frozen representations — arXiv:1911.01102.
- **PCA / t-SNE** visualization of hidden layers — Mohamed, Hinton & Penn (ICASSP 2012).
- **Generator-constrained (GAN/VAE) class visualization** — arXiv:1612.00005.
- **Decision-tree / random-forest interpretability** as the born-interpretable baseline.

Try swapping MNIST for Fashion-MNIST, or point these methods at your own classifier — the code is model-agnostic where it matters.